In [ ]:
import pandas as pd
import numpy as np

## Yield Prediction using Gradient Boosting and PyTorch

### 1. Data Loading and Initial Exploration

First, we'll load the dataset and take a look at its structure, data types, and a few sample rows.

In [ ]:
try:
    df = pd.read_csv('/content/production_unified_imputed.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: The file '/content/production_unified_imputed.csv' was not found. Please ensure it's uploaded correctly.")
    df = None


if df is not None:
    print("\nFirst 5 rows of the dataset:")
    display(df.head())

    print("\nDataFrame Info:")
    df.info()

Dataset loaded successfully.

First 5 rows of the dataset:


,crop,year,state,district,season,area,production,yield,data_source,annual_rainfall,fertilizer,pesticide,crop_type,area_unit,production_unit,yield_unit
0,wheat,2014,uttar pradesh,ghaziabad,Rabi,26619.0,86645.0,3.255,area_production,1247.199725,1.248187e+06,2438.165184,cereals,Hectare,Tonnes,Tonnes/Hectare
1,urad,2014,uttar pradesh,ghaziabad,Summer,22.0,12.0,0.545,area_production,1247.339375,1.235179e+06,2420.293698,pulses,Hectare,Tonnes,Tonnes/Hectare
2,urad,2014,uttar pradesh,ghaziabad,Kharif,19.0,10.0,0.526,area_production,1247.339396,1.235177e+06,2420.290923,pulses,Hectare,Tonnes,Tonnes/Hectare
3,sugarcane,2014,uttar pradesh,ghaziabad,Kharif,11136.0,750076.0,67.356,area_production,1247.223783,1.245441e+06,2437.372883,sugar,Hectare,Tonnes,Tonnes/Hectare
4,rice,2014,uttar pradesh,ghaziabad,Kharif,8650.0,23727.0,2.743,area_production,1247.278389,1.240637e+06,2428.557661,cereals,Hectare,Tonnes,Tonnes/Hectare



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440962 entries, 0 to 440961
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   crop             440962 non-null  object 
 1   year             440962 non-null  int64  
 2   state            440962 non-null  object 
 3   district         440962 non-null  object 
 4   season           440962 non-null  object 
 5   area             440962 non-null  float64
 6   production       440962 non-null  float64
 7   yield            440962 non-null  float64
 8   data_source      440962 non-null  object 
 9   annual_rainfall  440962 non-null  float64
 10  fertilizer       440962 non-null  float64
 11  pesticide        440962 non-null  float64
 12  crop_type        440962 non-null  object 
 13  area_unit        440962 non-null  object 
 14  production_unit  440962 non-null  object 
 15  yield_unit       440962 non-null  object 
dtypes: float64(6), int64(

### 2. Data Preprocessing

Now, let's prepare the data for modeling. This typically involves:
- Identifying categorical and numerical features.
- Encoding categorical features using techniques like one-hot encoding.
- Splitting the data into training and testing sets.

In [ ]:
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

if 'yield' in numerical_features:
    numerical_features.remove('yield')

print(f"Categorical features: {categorical_features}")
print(f"Numerical features (excluding target 'yield'): {numerical_features}")

df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

print("\nDataFrame after one-hot encoding:")
display(df_encoded.head())
print(f"New shape of DataFrame: {df_encoded.shape}")

Categorical features: ['crop', 'state', 'district', 'season', 'data_source', 'crop_type', 'area_unit', 'production_unit', 'yield_unit']
Numerical features (excluding target 'yield'): ['year', 'area', 'production', 'annual_rainfall', 'fertilizer', 'pesticide']

DataFrame after one-hot encoding:


,year,area,production,yield,annual_rainfall,fertilizer,pesticide,crop_arcanut (processed),crop_arecanut,crop_arhar/tur,...,data_source_des_district,crop_type_drugs and narcotics,crop_type_fiber crops,crop_type_fruits,crop_type_oilseeds,crop_type_plantation crops,crop_type_pulses,crop_type_spices,crop_type_sugar,crop_type_vegetable
0,2014,26619.0,86645.0,3.255,1247.199725,1.248187e+06,2438.165184,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2014,22.0,12.0,0.545,1247.339375,1.235179e+06,2420.293698,False,False,False,...,False,False,False,False,False,False,True,False,False,False
2,2014,19.0,10.0,0.526,1247.339396,1.235177e+06,2420.290923,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,2014,11136.0,750076.0,67.356,1247.223783,1.245441e+06,2437.372883,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,2014,8650.0,23727.0,2.743,1247.278389,1.240637e+06,2428.557661,False,False,False,...,False,False,False,False,False,False,False,False,False,False


New shape of DataFrame: (440962, 926)


### 3. Splitting Data into Training and Testing Sets

Now, we'll define our features (X) and target variable (y), and then split them into training and testing sets to evaluate our models effectively.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('yield', axis=1)
y = df_encoded['yield']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (352769, 925)
Shape of X_test: (88193, 925)
Shape of y_train: (352769,)
Shape of y_test: (88193,)


### 4. Preparing Data for PyTorch

To use PyTorch, we need to convert our numpy arrays (from pandas DataFrames/Series) into PyTorch tensors. We'll also create `DataLoader` objects for efficient batch processing during training.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader


X_train_np = X_train.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)
y_train_np = y_train.values.astype(np.float32)
y_test_np = y_test.values.astype(np.float32)

X_train_tensor = torch.tensor(X_train_np)
X_test_tensor = torch.tensor(X_test_np)
y_train_tensor = torch.tensor(y_train_np).unsqueeze(1)
y_test_tensor = torch.tensor(y_test_np).unsqueeze(1)

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nNumber of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

X_train_tensor shape: torch.Size([352769, 925])
y_train_tensor shape: torch.Size([352769, 1])

Number of training batches: 5513
Number of testing batches: 1379


### 5. Define PyTorch Model Architecture

Now we will define a simple Multi-Layer Perceptron (MLP) using PyTorch's `nn.Module` for our yield prediction task. Since this is a regression problem, the output layer will have a single neuron without an activation function, suitable for predicting continuous values.

In [ ]:
import torch.nn as nn
import torch.optim as optim

class YieldPredictor(nn.Module):
    def __init__(self, input_size):
        super(YieldPredictor, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, 64)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.2)
        self.fc4 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        return x

input_size = X_train_tensor.shape[1]
model = YieldPredictor(input_size)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)
print(f"\nLoss function: {criterion}")
print(f"Optimizer: {optimizer}")

YieldPredictor(
  (fc1): Linear(in_features=925, out_features=256, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (relu3): ReLU()
  (dropout3): Dropout(p=0.2, inplace=False)
  (fc4): Linear(in_features=64, out_features=1, bias=True)
)

Loss function: MSELoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


### 6. Train the PyTorch Model

We will now train the defined `YieldPredictor` model using our `train_loader`. We'll monitor the training loss and evaluate the model's performance on the test set after training.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Using device: {device}")

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

print("\nTraining finished!")

Using device: cpu
Epoch 1/10, Loss: 73900670.0026
Epoch 2/10, Loss: 12959.2578
Epoch 3/10, Loss: 38639.6320
Epoch 4/10, Loss: 602.3909
Epoch 5/10, Loss: 557.1156
Epoch 6/10, Loss: 555.4332
Epoch 7/10, Loss: 562.7984
Epoch 8/10, Loss: 557.8910
Epoch 9/10, Loss: 554.2515
Epoch 10/10, Loss: 558.4970

Training finished!


### 7. Evaluate the PyTorch Model

After training, we will evaluate the model's performance on the unseen test dataset using metrics like Mean Squared Error (MSE) and Root Mean Squared Error (RMSE).

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

predictions = np.array(predictions).flatten()
true_labels = np.array(true_labels).flatten()

mse = mean_squared_error(true_labels, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(true_labels, predictions)

print(f"\nModel Evaluation on Test Set:")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2): {r2:.4f}")


Model Evaluation on Test Set:
Mean Squared Error (MSE): 494.2927
Root Mean Squared Error (RMSE): 22.2327
R-squared (R2): -0.0000


### 8. Yield Prediction with Gradient Boosting (XGBoost)

Since the PyTorch model's performance was low, let's now implement a gradient boosting model, specifically XGBoost, which is known for its strong performance on tabular data. We will train it on the same training data and evaluate its performance.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost Regressor...")

xgb_model.fit(X_train, y_train)

print("XGBoost training complete. Evaluating on test set...")

xgb_predictions = xgb_model.predict(X_test)

mse_xgb = mean_squared_error(y_test, xgb_predictions)
rmse_xgb = np.sqrt(mse_xgb)
r2_xgb = r2_score(y_test, xgb_predictions)

print(f"\nXGBoost Model Evaluation on Test Set:")
print(f"Mean Squared Error (MSE): {mse_xgb:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_xgb:.4f}")
print(f"R-squared (R2): {r2_xgb:.4f}")

Training XGBoost Regressor...
XGBoost training complete. Evaluating on test set...

XGBoost Model Evaluation on Test Set:
Mean Squared Error (MSE): 56.4278
Root Mean Squared Error (RMSE): 7.5118
R-squared (R2): 0.8858


### 9. Summary and Comparison of Models

We've trained and evaluated both a PyTorch neural network and an XGBoost regressor for yield prediction. Let's compare their performance metrics.

In [ ]:
print("\n--- Model Performance Summary ---")
print(f"PyTorch Model:")
print(f"  Mean Squared Error (MSE): {mse:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"  R-squared (R2): {r2:.4f}")

print(f"\nXGBoost Model:")
print(f"  Mean Squared Error (MSE): {mse_xgb:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_xgb:.4f}")
print(f"  R-squared (R2): {r2_xgb:.4f}")

if r2_xgb > r2:
    print("\nConclusion: The XGBoost model performed significantly better than the simple PyTorch MLP for this yield prediction task.")
else:
    print("\nConclusion: The PyTorch MLP performed better or similarly to the XGBoost model for this yield prediction task.")

print("\nThis concludes the yield prediction analysis using gradient boosting and PyTorch. If you have further questions or want to explore other aspects, please let me know!")


--- Model Performance Summary ---
PyTorch Model:
  Mean Squared Error (MSE): 494.2927
  Root Mean Squared Error (RMSE): 22.2327
  R-squared (R2): -0.0000

XGBoost Model:
  Mean Squared Error (MSE): 56.4278
  Root Mean Squared Error (RMSE): 7.5118
  R-squared (R2): 0.8858

Conclusion: The XGBoost model performed significantly better than the simple PyTorch MLP for this yield prediction task.

This concludes the yield prediction analysis using gradient boosting and PyTorch. If you have further questions or want to explore other aspects, please let me know!


### 10. Exploring Other Gradient Boosting Methods and Hyperparameter Tuning

While XGBoost performed well, other gradient boosting libraries like LightGBM and CatBoost are also popular and can offer different performance characteristics, especially in terms of speed and handling of categorical features. Furthermore, hyperparameter tuning is crucial for extracting the best possible performance from any machine learning model.

Let's first briefly mention LightGBM and CatBoost, and then proceed with hyperparameter tuning for our XGBoost model.

#### A. Other Gradient Boosting Libraries (Brief Mention)

*   **LightGBM**: Developed by Microsoft, LightGBM is known for its high speed and efficiency. It uses a novel technique called Gradient-based One-Side Sampling (GOSS) to filter out data instances and Exclusive Feature Bundling (EFB) to bundle mutually exclusive features, leading to faster training.
*   **CatBoost**: Developed by Yandex, CatBoost excels at handling categorical features automatically. It uses a permutation-driven approach to overcome prediction shift and uses ordered boosting to avoid overfitting.

For brevity, we will focus on tuning the XGBoost model as it already demonstrated strong initial performance.

#### B. Hyperparameter Tuning for XGBoost

Hyperparameter tuning involves finding the best combination of parameters for a model that minimizes a chosen error metric (e.g., MSE). We'll use `RandomizedSearchCV` from `sklearn` to efficiently search through a defined range of hyperparameters. This method randomly samples combinations from the parameter space, which is generally more efficient than a full grid search for large search spaces.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

print("Starting Hyperparameter Tuning for XGBoost using RandomizedSearchCV...")

param_dist = {
    'n_estimators': stats.randint(100, 1000),
    'learning_rate': stats.loguniform(0.01, 0.3),
    'max_depth': stats.randint(3, 10),
    'subsample': stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'gamma': stats.uniform(0, 0.5),
    'reg_alpha': stats.loguniform(1e-5, 1),
    'reg_lambda': stats.loguniform(1e-5, 1)
}

random_search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("\nHyperparameter tuning complete.")
print(f"Best parameters found: {random_search.best_params_}")
print(f"Best cross-validation MSE: {-random_search.best_score_:.4f}")

best_xgb_model = random_search.best_estimator_

best_xgb_predictions = best_xgb_model.predict(X_test)

mse_best_xgb = mean_squared_error(y_test, best_xgb_predictions)
rmse_best_xgb = np.sqrt(mse_best_xgb)
r2_best_xgb = r2_score(y_test, best_xgb_predictions)

print(f"\nBest Tuned XGBoost Model Evaluation on Test Set:")
print(f"  Mean Squared Error (MSE): {mse_best_xgb:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_best_xgb:.4f}")
print(f"  R-squared (R2): {r2_best_xgb:.4f}")

Starting Hyperparameter Tuning for XGBoost using RandomizedSearchCV...
Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
